# STAR V1 - Spectral Transformer with Anchored Residuals
Single robust architecture with smart input preparation:
- log10E -> 32 log-spaced sinusoidal frequencies
- k_norm -> 16 Fourier features
- 4 anchor tokens (k_left, k_right, Ta_min, Ta_max)
- 23 descriptors as independent tokens
- Dual decoder: 64 Chebyshev + 32 cosine
- Sigmoid clamping, spectral norm, anti-oscillation penalty
- 7-seed ensemble

In [ ]:
EPOCHS = 1500
BATCH_SIZE = 16
LR = 2e-4
N_SEEDS = 7
CTX_NOISE = 0.03
N_CHEB = 64
N_COS = 32
D_MODEL = 384
N_LAYERS = 8
N_HEADS = 12
N_E_FREQ = 32
N_K_FREQ = 16
LAM_SPEC = 5e-5
LAM_SMOOTH = 5e-4

In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--index-url', 'https://download.pytorch.org/whl/cu121',
                       'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
AUX_DIR = list(INPUT.rglob('combined_data.csv'))[0].parent
CODE_DIR = list(INPUT.rglob('scripts'))[0].parent
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'TaylorCouetteML'
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(CODE_DIR, REPO_DIR)
os.chdir(REPO_DIR)
INPUT_CSV = REPO_DIR / 'data' / 'Input' / 'combined_data.csv'
INPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUX_DIR / 'combined_data.csv', INPUT_CSV)
print('Setup ready, REPO_DIR =', REPO_DIR)

In [ ]:
OUT_DIR = WORK / 'runs' / 'star_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)
args = [sys.executable, 'scripts/train_star_pro.py',
        '--epochs', str(EPOCHS),
        '--batch', str(BATCH_SIZE),
        '--lr', str(LR),
        '--n_seeds', str(N_SEEDS),
        '--ctx_noise', str(CTX_NOISE),
        '--n_cheb', str(N_CHEB),
        '--n_cos', str(N_COS),
        '--d_model', str(D_MODEL),
        '--n_layers', str(N_LAYERS),
        '--n_heads', str(N_HEADS),
        '--n_E_freq', str(N_E_FREQ),
        '--n_k_freq', str(N_K_FREQ),
        '--lam_spec', str(LAM_SPEC),
        '--lam_smooth', str(LAM_SMOOTH),
        '--out_root', str(OUT_DIR)]
print('>>>', ' '.join(args))
t0 = time.time()
rc = subprocess.call(args, cwd=str(REPO_DIR))
print(f'<<< exit={rc} elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'training failed'

In [ ]:
for root, dirs, files in os.walk(OUT_DIR):
    for f in files:
        p = Path(root) / f
        if p.suffix in ['.pt', '.pth', '.json', '.csv', '.npz']:
            print(p.relative_to(WORK), f'({p.stat().st_size/1e6:.2f} MB)')